# PatchCore on MVTec AD — Kaggle GPU (final)

This notebook runs the Anomalib PatchCore implementation on all 15 MVTec AD categories. It is designed for the Kaggle dataset input `anrdisded/mvtecac` containing `mvtec_anomaly_detection.tar.xz`.

**Execution order:** install and verify packages → locate/extract/validate data → configure PatchCore → run a `bottle` pilot → run all categories → create per-category metrics, aggregate CSVs, and a ZIP.

PatchCore does not use ordinary gradient-descent epochs. Its `fit()` stage extracts features from normal training images and builds a representative memory bank; `test()` evaluates image-level and pixel-level anomaly detection.

This notebook uses Anomalib 2.7.0 rather than the 2.6.2 version in the failed run. The 2.7.0 release includes a PatchCore coreset-selection loop optimization. The observed `RecursionError` from the previous run was not accompanied by a traceback, so its exact root cause cannot be proven from that run; this notebook changes the library version and captures full tracebacks if anything still fails.


References: [Anomalib 2.7.0 release notes](https://github.com/open-edge-platform/anomalib/releases) · [PatchCore reference](https://anomalib.readthedocs.io/en/latest/markdown/guides/reference/models/image/patchcore.html) · [MVTecAD datamodule API](https://anomalib.readthedocs.io/en/lib-v2.7.0/markdown/guides/reference/data/datamodules/image/mvtecad.html).

## 1. Kaggle setup

In Kaggle, enable **GPU** under Notebook Settings → Accelerator and attach the MVTec archive through **Add Input**. One T4 GPU is used per category run, even when Kaggle exposes two T4s. Internet access may be needed once to download the ImageNet-pretrained backbone weights.

Start this notebook in a fresh Kaggle session. Run the notebook from the top; do not start from the category loop.


In [ ]:
%pip install -q --no-cache-dir torch==2.11.0 torchvision==0.26.0 --index-url https://download.pytorch.org/whl/cu130
%pip install -q --no-cache-dir anomalib==2.7.0

In [ ]:
import os
import platform
from importlib.metadata import version
os.environ["TQDM_DISABLE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import json
import random
import shutil
import tarfile
import time
import traceback
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchvision
import anomalib
from anomalib.data import MVTecAD
from anomalib.engine import Engine
from anomalib.models import Patchcore

print("Python:", os.sys.version)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Anomalib:", anomalib.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if anomalib.__version__ != "2.7.0":
    raise RuntimeError(f"Expected Anomalib 2.7.0, found {anomalib.__version__}. Restart the Kaggle session and run from the top.")

if not torch.cuda.is_available():
    raise RuntimeError("GPU is not enabled. Select a GPU accelerator in Kaggle Settings, then restart and run from the top.")

for gpu_index in range(torch.cuda.device_count()):
    print(f"GPU {gpu_index}:", torch.cuda.get_device_name(gpu_index))

torch.set_num_threads(min(4, os.cpu_count() or 4))
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)


## 2. Locate and validate MVTec AD

The data input is read-only. The archive is extracted to `/kaggle/working`, which is writable. If the archive was already extracted during this session, the existing valid root is reused.


In [ ]:
CATEGORIES = [
    "bottle", "cable", "capsule", "carpet", "grid",
    "hazelnut", "leather", "metal_nut", "pill", "screw",
    "tile", "toothbrush", "transistor", "wood", "zipper"
]

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
EXTRACT_DIR = WORK_ROOT / "mvtec_extracted"
OUTPUT_ROOT = WORK_ROOT / "patchcore_runs_anomalib_2_7_0"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

environment = {
    "python": os.sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpu": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "packages": {
        name: version(name)
        for name in ["anomalib", "numpy", "pandas", "scikit-learn", "scikit-image", "opencv-python-headless", "matplotlib", "pillow"]
    },
}
(OUTPUT_ROOT / "environment.json").write_text(json.dumps(environment, indent=2))
print("Saved environment manifest:", OUTPUT_ROOT / "environment.json")


def is_mvtec_root(path):
    path = Path(path)
    if not path.is_dir():
        return False
    for category_name in CATEGORIES:
        category_root = path / category_name
        if not (category_root / "train" / "good").is_dir():
            return False
        if not (category_root / "test").is_dir():
            return False
        if not (category_root / "ground_truth").is_dir():
            return False
    return True


def find_existing_root(search_root):
    search_root = Path(search_root)
    if is_mvtec_root(search_root):
        return search_root
    for candidate in search_root.rglob("bottle"):
        if candidate.is_dir() and candidate.name == "bottle":
            possible_root = candidate.parent
            if is_mvtec_root(possible_root):
                return possible_root
    return None


DATA_ROOT = find_existing_root(EXTRACT_DIR) if EXTRACT_DIR.exists() else None
if DATA_ROOT is None:
    DATA_ROOT = find_existing_root(INPUT_ROOT)

if DATA_ROOT is None:
    archives = sorted(INPUT_ROOT.rglob("mvtec_anomaly_detection.tar.xz"))
    if not archives:
        visible_files = [str(path) for path in INPUT_ROOT.rglob("*") if path.is_file()]
        raise FileNotFoundError(
            "Could not find a valid extracted MVTec root or mvtec_anomaly_detection.tar.xz under /kaggle/input. "
            "Attach the dataset through Add Input. Visible files: " + " | ".join(visible_files[:40])
        )

    archive_path = archives[0]
    archive_size = archive_path.stat().st_size
    free_bytes = shutil.disk_usage(WORK_ROOT).free
    print("Using archive:", archive_path)
    print("Archive size (GB):", round(archive_size / 1024**3, 2))
    print("Free working space (GB):", round(free_bytes / 1024**3, 2))

    if free_bytes < archive_size * 2.2:
        raise RuntimeError("Insufficient free disk space for safe extraction. Free space or use an already-extracted MVTec dataset input.")

    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    print("Extracting MVTec archive...")
    with tarfile.open(archive_path, mode="r:xz") as archive:
        archive.extractall(path=EXTRACT_DIR, filter="data")

    DATA_ROOT = find_existing_root(EXTRACT_DIR)

if DATA_ROOT is None or not is_mvtec_root(DATA_ROOT):
    raise FileNotFoundError("MVTec extraction completed, but the expected 15-category folder structure was not found.")

print("MVTec root:", DATA_ROOT)

category_counts = []
for category_name in CATEGORIES:
    train_images = list((DATA_ROOT / category_name / "train" / "good").glob("*.png"))
    test_images = list((DATA_ROOT / category_name / "test").glob("*/*.png"))
    mask_images = list((DATA_ROOT / category_name / "ground_truth").glob("*/*.png"))
    if not train_images or not test_images:
        raise RuntimeError(f"Missing training or test images for category {category_name}.")
    category_counts.append({
        "category": category_name,
        "normal_train_images": len(train_images),
        "test_images": len(test_images),
        "ground_truth_masks": len(mask_images)
    })

counts_df = pd.DataFrame(category_counts)
display(counts_df)
print("All 15 categories passed the folder and image checks.")


## 3. PatchCore configuration

The configuration retains the intended PatchCore setup: ImageNet-pretrained `wide_resnet50_2`, features from `layer2` and `layer3`, 10% coreset sampling, and 9 nearest neighbours. `num_workers=0` avoids worker-process shutdown issues in notebook runtimes. Training and evaluation use a single GPU.


In [ ]:
BACKBONE = "wide_resnet50_2"
LAYERS = ["layer2", "layer3"]
CORESET_RATIO = 0.1
NUM_NEIGHBORS = 9
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
NUM_WORKERS = 0
RANDOM_SEED = 42


def build_datamodule(category):
    return MVTecAD(
        root=str(DATA_ROOT),
        category=category,
        train_batch_size=TRAIN_BATCH_SIZE,
        eval_batch_size=EVAL_BATCH_SIZE,
        num_workers=NUM_WORKERS,
        seed=RANDOM_SEED
    )


def build_model():
    return Patchcore(
        backbone=BACKBONE,
        layers=LAYERS,
        pre_trained=True,
        coreset_sampling_ratio=CORESET_RATIO,
        num_neighbors=NUM_NEIGHBORS
    )


def build_engine(category):
    category_root = OUTPUT_ROOT / category
    category_root.mkdir(parents=True, exist_ok=True)
    return Engine(
        default_root_dir=str(category_root),
        accelerator="gpu",
        devices=1,
        max_epochs=1,
        logger=False,
        enable_checkpointing=True,
        enable_progress_bar=False,
        enable_model_summary=False
    )

print("Configuration ready.")
print("Checkpointing: enabled (required by Anomalib ModelCheckpoint callback).")
print("Backbone:", BACKBONE)
print("Feature layers:", LAYERS)
print("Coreset ratio:", CORESET_RATIO)
print("Nearest neighbours:", NUM_NEIGHBORS)
print("DataLoader workers:", NUM_WORKERS)


## 4. Result-saving utilities and one-category pilot

The pilot runs `bottle` first. The notebook will stop here if the pilot fails, and the full traceback is saved to the category-specific `failure.json`. This prevents spending time on all 15 categories when the core training/evaluation path is still broken.


In [ ]:
def make_jsonable(value):
    if isinstance(value, dict):
        return {str(key): make_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_jsonable(item) for item in value]
    if isinstance(value, torch.Tensor):
        value = value.detach().cpu()
        return value.item() if value.numel() == 1 else value.tolist()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return str(value)


def save_json_atomic(path, payload):
    path = Path(path)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with open(temporary_path, "w") as file:
        json.dump(make_jsonable(payload), file, indent=2)
    temporary_path.replace(path)


def load_completed_record(category):
    result_path = OUTPUT_ROOT / category / "test_results.json"
    if not result_path.exists():
        return None
    try:
        with open(result_path) as file:
            record = json.load(file)
        if record.get("completed") is True:
            return record
    except Exception:
        return None
    return None


def run_category(category, force_rerun=False):
    existing_record = load_completed_record(category)
    if existing_record is not None and not force_rerun:
        print("Skipping completed category:", category)
        return existing_record

    category_root = OUTPUT_ROOT / category
    category_root.mkdir(parents=True, exist_ok=True)
    result_path = category_root / "test_results.json"
    started = time.time()
    datamodule = None
    model = None
    engine = None

    print("\n" + "=" * 72)
    print("PATCHCORE CATEGORY:", category)
    print("=" * 72)

    try:
        datamodule = build_datamodule(category)
        model = build_model()
        engine = build_engine(category)

        print("Starting PatchCore fit: feature extraction and memory-bank construction.")
        engine.fit(model=model, datamodule=datamodule)
        print("PatchCore fit completed. Starting test evaluation.")
        test_results = engine.test(model=model, datamodule=datamodule)

        if test_results is None:
            raise RuntimeError("Engine.test returned None; no evaluation metrics were produced.")

        record = {
            "category": category,
            "completed": True,
            "status": "completed",
            "runtime_minutes": round((time.time() - started) / 60, 2),
            "test_results": make_jsonable(test_results),
            "configuration": {
                "anomalib_version": anomalib.__version__,
                "pytorch_version": torch.__version__,
                "backbone": BACKBONE,
                "layers": LAYERS,
                "coreset_sampling_ratio": CORESET_RATIO,
                "num_neighbors": NUM_NEIGHBORS,
                "train_batch_size": TRAIN_BATCH_SIZE,
                "eval_batch_size": EVAL_BATCH_SIZE,
                "num_workers": NUM_WORKERS
            }
        }
        save_json_atomic(result_path, record)
        stale_failure_path = category_root / "failure.json"
        if stale_failure_path.exists():
            stale_failure_path.unlink()
        print(json.dumps(record, indent=2))
        return record

    except Exception as error:
        failure = {
            "category": category,
            "error_type": type(error).__name__,
            "error": str(error),
            "traceback": traceback.format_exc(),
            "runtime_minutes_before_failure": round((time.time() - started) / 60, 2)
        }
        failure_path = category_root / "failure.json"
        save_json_atomic(failure_path, failure)
        print("FAILED:", category)
        print(failure["traceback"])
        return {"category": category, "completed": False, "status": "failed", "failure_file": str(failure_path)}

    finally:
        if engine is not None:
            try:
                del engine
            except Exception:
                pass
        if model is not None:
            try:
                del model
            except Exception:
                pass
        if datamodule is not None:
            try:
                del datamodule
            except Exception:
                pass
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


pilot_result = run_category("bottle", force_rerun=False)
if pilot_result.get("completed") is not True:
    raise RuntimeError("The bottle pilot did not complete. Read /kaggle/working/patchcore_runs_anomalib_2_7_0/bottle/failure.json before proceeding.")

print("Pilot passed. Proceed to all 15 categories.")


## 5. Run all 15 categories

Completed categories are skipped on rerun. Each category writes its result immediately, so successful results survive an interruption within the current Kaggle session. Failures include the full traceback and do not prevent the remaining categories from running after the pilot has passed.


In [ ]:
all_category_results = []
failed_categories = []

for category in CATEGORIES:
    record = run_category(category, force_rerun=False)
    if record.get("completed") is True:
        all_category_results.append(record)
    else:
        failed_categories.append(record)

    save_json_atomic(OUTPUT_ROOT / "all_categories_results.json", all_category_results)
    save_json_atomic(OUTPUT_ROOT / "failed_categories.json", failed_categories)

    completed_records = [load_completed_record(name) for name in CATEGORIES]
    completed_records = [record for record in completed_records if record is not None]
    save_json_atomic(OUTPUT_ROOT / "completed_categories.json", completed_records)

print("\nRun finished.")
print("Completed:", len(completed_records), "of", len(CATEGORIES))
print("Failed this run:", len(failed_categories))
print("Output directory:", OUTPUT_ROOT)


## 6. Aggregate per-category metrics

The metrics extractor accepts either a dictionary or Anomalib's list-of-dictionaries test output. It does not invent metrics: only numeric values returned by the test evaluator are written. The category CSV contains one row per completed category; the summary CSV contains the macro mean across completed categories.


In [ ]:
def flatten_numeric(value, prefix=""):
    flattened = {}
    if isinstance(value, dict):
        for key, item in value.items():
            name = f"{prefix}.{key}" if prefix else str(key)
            flattened.update(flatten_numeric(item, name))
    elif isinstance(value, list):
        if len(value) == 1 and isinstance(value[0], dict):
            flattened.update(flatten_numeric(value[0], prefix))
        else:
            for index, item in enumerate(value):
                name = f"{prefix}.{index}" if prefix else str(index)
                flattened.update(flatten_numeric(item, name))
    elif isinstance(value, (int, float, np.integer, np.floating)) and not isinstance(value, bool):
        flattened[prefix] = float(value)
    return flattened


metric_rows = []
for category in CATEGORIES:
    record = load_completed_record(category)
    if record is None:
        continue
    row = {
        "category": category,
        "runtime_minutes": record.get("runtime_minutes", np.nan)
    }
    row.update(flatten_numeric(record.get("test_results", {})))
    metric_rows.append(row)

metrics_df = pd.DataFrame(metric_rows)
metrics_path = OUTPUT_ROOT / "all_categories_metrics.csv"
summary_path = OUTPUT_ROOT / "aggregate_summary.csv"

if not metrics_df.empty:
    metrics_df.to_csv(metrics_path, index=False)
    metric_columns = [column for column in metrics_df.columns if column not in {"category", "runtime_minutes"}]
    summary_record = {
        "statistic": "macro_mean_across_completed_categories",
        "completed_categories": int(len(metrics_df)),
        "total_runtime_minutes": float(pd.to_numeric(metrics_df["runtime_minutes"], errors="coerce").sum()),
        "mean_runtime_minutes": float(pd.to_numeric(metrics_df["runtime_minutes"], errors="coerce").mean())
    }
    for column in metric_columns:
        summary_record[column] = float(pd.to_numeric(metrics_df[column], errors="coerce").mean())
    pd.DataFrame([summary_record]).to_csv(summary_path, index=False)
    display(metrics_df)
    display(pd.DataFrame([summary_record]))
    print("Saved:", metrics_path)
    print("Saved:", summary_path)
else:
    pd.DataFrame(columns=["category", "runtime_minutes"]).to_csv(metrics_path, index=False)
    pd.DataFrame([{"statistic": "macro_mean_across_completed_categories", "completed_categories": 0}]).to_csv(summary_path, index=False)
    print("No completed category results are available yet.")

failure_rows = []
for category in CATEGORIES:
    failure_path = OUTPUT_ROOT / category / "failure.json"
    if failure_path.exists() and load_completed_record(category) is None:
        with open(failure_path) as file:
            failure_rows.append(json.load(file))
if failure_rows:
    pd.DataFrame(failure_rows).to_csv(OUTPUT_ROOT / "failures.csv", index=False)
    print("Saved:", OUTPUT_ROOT / "failures.csv")


## 7. Package results for download

The ZIP includes per-category JSON results, failure tracebacks (if any), aggregate CSVs, and Anomalib output files written under the run directory.


In [ ]:
archive_path = WORK_ROOT / "PatchCore_MVTec_Kaggle_results.zip"

with zipfile.ZipFile(archive_path, mode="w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in OUTPUT_ROOT.rglob("*"):
        if file_path.is_file() and file_path != archive_path:
            archive.write(file_path, file_path.relative_to(OUTPUT_ROOT))

archive_size_mb = archive_path.stat().st_size / (1024 ** 2)
print("Results archive:", archive_path)
print("Archive size (MB):", round(archive_size_mb, 2))
print("Completed category count:", sum(load_completed_record(category) is not None for category in CATEGORIES))
print("Open the Kaggle notebook Output panel after saving a notebook version to download the ZIP and CSV files.")


## Notes and experiment interpretation

- MVTec AD training images under `train/good` are the normal-only training set. Defective test images and ground-truth masks are used only for evaluation.
- PatchCore's `fit()` builds a memory bank of normal patch embeddings. It is not a conventional multi-epoch neural-network optimization process.
- `image_AUROC` measures image-level ranking; `pixel_AUROC` measures pixel-level localization. AP/F1 columns are included only if the Anomalib evaluator returns them.
- Use the same input size, transforms, backbone, feature layers, coreset ratio, and evaluation procedure when comparing PatchCore with the AE/PCA models.
- This notebook is pinned to Anomalib 2.7.0. The 2.7.0 release notes specify a PyTorch minimum of 2.10 and include a coreset-selection loop optimization. The notebook checks the installed version before using it.
